## Silver — Custos, Pesquisas de Satisfação e Chamados_Hora

### 1. Configuração da origem dos dados

Utilizamos as tabelas Delta produzidas pelo notebook de ingestão
Bronze no catálogo vcredit_medallion.

A configuração centraliza os nomes das quatro tabelas utilizadas:
Custos, Pesquisa de Satisfação, Chamados_Hora e Chamados.

A referência de Chamados é utilizada para verificar correspondências
e comparar os identificadores de cliente e os tempos informados.

A execução depende da disponibilidade dessas tabelas no ambiente.

In [0]:
catalogo = "vcredit_medallion"

esquema_bronze = f"{catalogo}.bronze"
esquema_silver = f"{catalogo}.silver"

tabelas_bronze = {
    "custos": f"{esquema_bronze}.tb_custos",
    "pesquisa_satisfacao": (
        f"{esquema_bronze}.tb_pesquisa_satisfacao"
    ),
    "chamados_hora": f"{esquema_bronze}.tb_chamados_hora",
    "chamados": f"{esquema_bronze}.tb_chamados"
}

print(f"Origem: {esquema_bronze}")
print(f"Destino da Silver: {esquema_silver}")

### 2. Leitura das tabelas Bronze

Lemos as tabelas Bronze registradas no catálogo, mantendo os nomes
dos DataFrames utilizados nos tratamentos desenvolvidos.

Padronizamos os nomes das colunas para minúsculas e verificamos a
presença dos campos necessários.

Os campos de negócio devem permanecer como texto, conforme a
ingestão Bronze, para preservar os valores recebidos e permitir
auditar as conversões na Silver.

Selecionamos explicitamente as colunas de negócio de cada base.
O metadado ingestion_datetime permanece na Bronze e não participa
das transformações ou das comparações dos registros de negócio.

Essa seleção preserva todas as linhas da origem. A verificação
e o tratamento de duplicidades ocorrem nas etapas seguintes.

In [0]:
def ler_bronze(nome_base, colunas_esperadas):
    nome_tabela = tabelas_bronze[nome_base]

    dados = spark.read.table(nome_tabela)

    nomes_colunas = [
        coluna.strip().lower()
        for coluna in dados.columns
    ]

    if len(nomes_colunas) != len(set(nomes_colunas)):
        raise ValueError(
            f"Colunas repetidas após padronização em {nome_tabela}."
        )

    dados = dados.toDF(*nomes_colunas)

    colunas_ausentes = sorted(
        set(colunas_esperadas) - set(dados.columns)
    )

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes em {nome_tabela}: "
            f"{colunas_ausentes}"
        )

    dados = dados.select(*colunas_esperadas)

    tipos_incompativeis = [
        coluna
        for coluna, tipo in dados.dtypes
        if tipo != "string"
    ]

    if tipos_incompativeis:
        raise ValueError(
            f"Campos de negócio devem permanecer como texto em "
            f"{nome_tabela}: {tipos_incompativeis}"
        )

    return dados


df_custos_origem = ler_bronze(
    "custos",
    [
        "id_custo",
        "id_chamado",
        "custo"
    ]
)

df_pesquisa_satisfacao_origem = ler_bronze(
    "pesquisa_satisfacao",
    [
        "id_pesquisa",
        "id_chamado",
        "nota_atendimento"
    ]
)

df_chamados_hora_origem = ler_bronze(
    "chamados_hora",
    [
        "id_chamado",
        "id_cliente",
        "hora_abertura_chamado",
        "hora_inicio_atendimento",
        "hora_finalizacao_atendimento"
    ]
)

df_chamados_referencia = ler_bronze(
    "chamados",
    [
        "id_chamado",
        "id_cliente",
        "motivo",
        "canal",
        "resolvido",
        "hora_abertura_chamado",
        "hora_inicio_atendimento",
        "hora_finalizacao_atendimento",
        "tempo_espera",
        "tempo_atendimento",
        "id_atendente"
    ]
)

### 3. Conferência das contagens de origem

Registramos a quantidade de linhas e colunas de cada base antes
dos tratamentos. Essas contagens servirão de referência para verificar
as alterações produzidas nas próximas etapas.

As quantidades representam os registros recebidos, incluindo eventuais
duplicidades. A quantidade de linhas da origem não equivale
necessariamente à quantidade de identificadores distintos.

In [0]:
resumo_leitura = [
    (
        "custos",
        df_custos_origem.count(),
        len(df_custos_origem.columns)
    ),
    (
        "pesquisa_satisfacao",
        df_pesquisa_satisfacao_origem.count(),
        len(df_pesquisa_satisfacao_origem.columns)
    ),
    (
        "chamados_hora",
        df_chamados_hora_origem.count(),
        len(df_chamados_hora_origem.columns)
    ),
    (
        "chamados_referencia",
        df_chamados_referencia.count(),
        len(df_chamados_referencia.columns)
    )
]

df_resumo_leitura = spark.createDataFrame(
    resumo_leitura,
    ["base", "quantidade_linhas", "quantidade_colunas"]
)

display(df_resumo_leitura)

### 4. Verificação de duplicidades em Custos

Consideramos duplicatas exatas os registros que apresentam os mesmos
valores de id_custo, id_chamado e custo. Mantemos uma ocorrência
de cada combinação e registramos a quantidade de repetições removidas.

Após essa remoção, verificamos se um mesmo id_custo ainda identifica
registros diferentes. Se isso ocorrer, interrompemos o tratamento
para investigar o conflito, sem escolher uma ocorrência arbitrariamente.

O id_chamado não será utilizado isoladamente para remover duplicidades,
pois um chamado pode possuir diferentes registros de custo.

In [0]:
from pyspark.sql import functions as sf

colunas_custos = ["id_custo", "id_chamado", "custo"]

quantidade_custos_origem = df_custos_origem.count()

df_custos_sem_duplicatas = df_custos_origem.dropDuplicates(
    colunas_custos
)

quantidade_custos_sem_duplicatas = (
    df_custos_sem_duplicatas.count()
)

df_conflitos_id_custo = (
    df_custos_sem_duplicatas
    .groupBy("id_custo")
    .count()
    .filter(sf.col("count") > 1)
)

quantidade_conflitos_id_custo = df_conflitos_id_custo.count()

resumo_duplicidades_custos = [
    ("linhas_origem", quantidade_custos_origem),
    ("linhas_sem_duplicatas", quantidade_custos_sem_duplicatas),
    (
        "repeticoes_exatas_removidas",
        quantidade_custos_origem - quantidade_custos_sem_duplicatas
    ),
    ("ids_com_conflito", quantidade_conflitos_id_custo)
]

display(
    spark.createDataFrame(
        resumo_duplicidades_custos,
        ["verificacao", "quantidade"]
    )
)

if quantidade_conflitos_id_custo > 0:
    raise ValueError(
        "Existem id_custo com registros diferentes. "
        "Investigue os conflitos antes de continuar."
    )

### 5. Conversão dos valores monetários

A coluna custo apresenta diferentes representações, como valores com
vírgula decimal, ponto decimal, prefixo R$ e sufixo reais.

Preservamos o conteúdo recebido em custo_original. Para conversão,
removemos somente os marcadores monetários reconhecidos, ajustamos
o separador decimal e aceitamos números com até duas casas decimais.

Utilizamos decimal(18,2) para representar valores monetários sem
os erros de aproximação associados ao ponto flutuante.

Conteúdos incompatíveis com esses formatos são convertidos para nulo
e identificados por custo_nao_convertido, sem atribuição de valor zero.

Custos negativos e custos iguais a zero são preservados. Os negativos
são sinalizados, pois a documentação disponível não esclarece se
representam ajustes legítimos ou erros de registro.

In [0]:
def converter_custos(dados):
    return (
        dados
        .withColumnRenamed("custo", "custo_original")
        .withColumn(
            "custo_texto",
            sf.lower(sf.trim(sf.col("custo_original")))
        )
        .withColumn(
            "custo_texto",
            sf.regexp_replace(
                sf.col("custo_texto"),
                r"^r\$\s*",
                ""
            )
        )
        .withColumn(
            "custo_texto",
            sf.regexp_replace(
                sf.col("custo_texto"),
                r"\s*reais$",
                ""
            )
        )
        .withColumn(
            "custo_texto",
            sf.regexp_replace(
                sf.trim(sf.col("custo_texto")),
                ",",
                "."
            )
        )
        .withColumn(
            "custo",
            sf.when(
                sf.col("custo_texto").rlike(
                    r"^[+-]?[0-9]+(?:\.[0-9]{1,2})?$"
                ),
                sf.expr(
                    "try_cast(custo_texto as decimal(18,2))"
                )
            )
        )
        .withColumn(
            "custo_nao_convertido",
            sf.col("custo").isNull()
        )
        .withColumn(
            "custo_negativo",
            sf.coalesce(
                sf.col("custo") < 0,
                sf.lit(False)
            )
        )
        .drop("custo_texto")
    )


df_custos_tratados = converter_custos(
    df_custos_sem_duplicatas
)

### 6. Conferência inicial dos custos tratados

Verificamos a quantidade de registros, as falhas de conversão,
os valores negativos, os valores iguais a zero e a soma dos custos.

A soma apresentada inclui os valores negativos preservados.
Ela constitui uma conferência técnica dos valores convertidos.
A interpretação dos negativos deverá ser alinhada com o grupo
antes da definição do indicador financeiro final.

Caso existam valores não convertidos, a soma corresponde somente
aos valores convertidos e exige investigação antes de ser utilizada.

In [0]:
df_resumo_custos = df_custos_tratados.agg(
    sf.count("*").alias("registros"),
    sf.sum(
        sf.col("custo_nao_convertido").cast("long")
    ).alias("custos_nao_convertidos"),
    sf.sum(
        sf.col("custo_negativo").cast("long")
    ).alias("custos_negativos"),
    sf.sum(
        sf.when(
            sf.col("custo") == 0,
            1
        ).otherwise(0)
    ).alias("custos_iguais_a_zero"),
    sf.sum("custo").alias("soma_custos_com_sinal")
)

display(df_resumo_custos)

### Amostra dos custos tratados

Exibimos cinco registros com o valor original, o valor convertido
e as sinalizações de qualidade para visualizar o tratamento monetário.

In [0]:
display(
    df_custos_tratados.select(
        "id_custo",
        "id_chamado",
        "custo_original",
        "custo",
        "custo_nao_convertido",
        "custo_negativo"
    ).limit(5)
)

### 7. Correspondência entre Custos e Chamados

Utilizamos uma lista de id_chamado distintos da referência de Chamados.
Essa lista evita que repetições existentes na referência multipliquem
os registros de custo durante o cruzamento.

O cruzamento preserva todos os custos e acrescenta chamado_encontrado,
indicando a existência de correspondência na referência.

Verificamos também quais chamados não possuem registro de custo.
Essa ausência é mantida como uma lacuna de informação, distinguindo-a
de um custo registrado com valor zero.

A quantidade de registros de custo deve permanecer igual após
o cruzamento.

In [0]:
df_ids_chamados = (
    df_chamados_referencia
    .select("id_chamado")
    .distinct()
)


def adicionar_correspondencia_chamados(dados):
    referencia = df_ids_chamados.withColumn(
        "chamado_encontrado",
        sf.lit(True)
    )

    return (
        dados
        .join(
            referencia,
            on="id_chamado",
            how="left"
        )
        .fillna({"chamado_encontrado": False})
    )


df_custos_com_correspondencia = (
    adicionar_correspondencia_chamados(df_custos_tratados)
)

quantidade_custos_antes = df_custos_tratados.count()

quantidade_custos_depois = (
    df_custos_com_correspondencia.count()
)

quantidade_custos_sem_chamado = (
    df_custos_com_correspondencia
    .filter(~sf.col("chamado_encontrado"))
    .count()
)

df_chamados_sem_custo = df_ids_chamados.join(
    df_custos_tratados.select("id_chamado").distinct(),
    on="id_chamado",
    how="left_anti"
)

resumo_correspondencia = [
    ("custos_antes_cruzamento", quantidade_custos_antes),
    ("custos_apos_cruzamento", quantidade_custos_depois),
    ("custos_sem_chamado", quantidade_custos_sem_chamado),
    ("chamados_distintos_referencia", df_ids_chamados.count()),
    ("chamados_sem_registro_custo", df_chamados_sem_custo.count())
]

display(
    spark.createDataFrame(
        resumo_correspondencia,
        ["verificacao", "quantidade"]
    )
)

if quantidade_custos_antes != quantidade_custos_depois:
    raise ValueError(
        "O cruzamento alterou a quantidade de registros de custo."
    )

### 8. Reconciliação dos totais de Custos

Aplicamos a mesma regra de conversão monetária à origem e aos registros
deduplicados, garantindo uma comparação consistente.

Calculamos separadamente o valor das repetições removidas.
Para cada combinação repetida de id_custo, id_chamado e custo,
multiplicamos o custo pela quantidade de ocorrências excedentes.

A soma da origem deve ser igual à soma após deduplicação acrescida
do valor dessas repetições.

Interrompemos a conferência financeira caso existam custos não
convertidos, evitando utilizar uma soma incompleta.

Os totais abrangem todo o arquivo e incluem os valores negativos
preservados. A apuração por período será realizada na análise de negócio.

In [0]:
df_custos_origem_convertidos = converter_custos(
    df_custos_origem
)

if (
    df_custos_origem_convertidos
    .filter(sf.col("custo_nao_convertido"))
    .limit(1)
    .count()
) > 0:
    raise ValueError(
        "Existem custos não convertidos. "
        "Investigue antes de utilizar os totais."
    )


def somar_custos(dados):
    resultado = dados.agg(
        sf.coalesce(
            sf.sum("custo"),
            sf.lit(0).cast("decimal(38,2)")
        ).alias("total")
    ).first()

    return resultado["total"]


df_grupos_custos_repetidos = (
    df_custos_origem
    .groupBy(*colunas_custos)
    .agg(sf.count("*").alias("ocorrencias"))
    .filter(sf.col("ocorrencias") > 1)
)

df_valores_repetidos = (
    converter_custos(df_grupos_custos_repetidos)
    .select(
        (
            sf.col("custo") * (sf.col("ocorrencias") - 1)
        ).alias("custo")
    )
)

total_custos_origem = somar_custos(
    df_custos_origem_convertidos
)

total_custos_sem_duplicatas = somar_custos(
    df_custos_com_correspondencia
)

total_repeticoes_removidas = somar_custos(
    df_valores_repetidos
)

diferenca_reconciliacao = (
    total_custos_origem
    - total_custos_sem_duplicatas
    - total_repeticoes_removidas
)

resumo_reconciliacao = [
    ("soma_origem", total_custos_origem),
    ("soma_sem_duplicatas", total_custos_sem_duplicatas),
    ("valor_repeticoes_removidas", total_repeticoes_removidas),
    ("diferenca_reconciliacao", diferenca_reconciliacao)
]

display(
    spark.createDataFrame(
        resumo_reconciliacao,
        "verificacao string, valor decimal(38,2)"
    )
)

if diferenca_reconciliacao != 0:
    raise ValueError(
        "Os totais de Custos não foram reconciliados."
    )

### 9. Reexecução das transformações de Custos

Reaplicamos a deduplicação, a conversão monetária e a verificação
de correspondência sobre a mesma origem.

Comparamos os resultados nos dois sentidos utilizando exceptAll,
que considera tanto os valores quanto a multiplicidade das linhas.
As colunas são alinhadas na mesma ordem antes da comparação.

O resultado esperado é ausência de diferenças entre as execuções,
considerando os mesmos arquivos e configurações.

Este teste verifica a estabilidade das transformações. A reexecução
da gravação será conferida quando as tabelas Silver forem publicadas.

In [0]:
df_custos_reexecucao = adicionar_correspondencia_chamados(
    converter_custos(
        df_custos_origem.dropDuplicates(colunas_custos)
    )
)

colunas_comparacao_custos = sorted(
    df_custos_com_correspondencia.columns
)

df_custos_primeira_execucao = (
    df_custos_com_correspondencia
    .select(*colunas_comparacao_custos)
)

df_custos_segunda_execucao = (
    df_custos_reexecucao
    .select(*colunas_comparacao_custos)
)

if (
    df_custos_primeira_execucao.schema
    != df_custos_segunda_execucao.schema
):
    raise ValueError(
        "A reexecução alterou a estrutura dos dados de Custos."
    )

linhas_ausentes_reexecucao = (
    df_custos_primeira_execucao
    .exceptAll(df_custos_segunda_execucao)
    .count()
)

linhas_adicionais_reexecucao = (
    df_custos_segunda_execucao
    .exceptAll(df_custos_primeira_execucao)
    .count()
)

display(
    spark.createDataFrame(
        [
            ("linhas_ausentes", linhas_ausentes_reexecucao),
            ("linhas_adicionais", linhas_adicionais_reexecucao)
        ],
        ["verificacao", "quantidade"]
    )
)

if (
    linhas_ausentes_reexecucao > 0
    or linhas_adicionais_reexecucao > 0
):
    raise ValueError(
        "Foram encontradas diferenças na reexecução de Custos."
    )

df_custos_silver = df_custos_com_correspondencia.select(
    "id_custo",
    "id_chamado",
    "custo",
    "custo_original",
    "custo_nao_convertido",
    "custo_negativo",
    "chamado_encontrado"
)

### 10. Inspeção de Pesquisas de Satisfação

A granularidade desta base é uma pesquisa, identificada por id_pesquisa.

Verificamos repetições exatas considerando id_pesquisa, id_chamado
e nota_atendimento. Após essa verificação, identificamos eventuais
conflitos de id_pesquisa.

Também contamos os chamados associados a mais de uma pesquisa.
Essas ocorrências são preservadas quando possuem identificadores
de pesquisa diferentes, pois a repetição de id_chamado isoladamente
não comprova duplicidade.

Inspecionamos a distribuição das notas antes de aplicar as regras
da escala de 1 a 5.

Na exibição de diagnóstico, valores nulos e textos vazios recebem
rótulos próprios para facilitar sua identificação.

In [0]:
colunas_pesquisas = [
    "id_pesquisa",
    "id_chamado",
    "nota_atendimento"
]

df_pesquisas_distintas = (
    df_pesquisa_satisfacao_origem
    .dropDuplicates(colunas_pesquisas)
)

quantidade_pesquisas_origem = (
    df_pesquisa_satisfacao_origem.count()
)

quantidade_pesquisas_distintas = (
    df_pesquisas_distintas.count()
)

quantidade_conflitos_pesquisa = (
    df_pesquisas_distintas
    .groupBy("id_pesquisa")
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

quantidade_chamados_multiplas_pesquisas = (
    df_pesquisas_distintas
    .groupBy("id_chamado")
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

resumo_pesquisas = [
    ("linhas_origem", quantidade_pesquisas_origem),
    (
        "repeticoes_exatas",
        quantidade_pesquisas_origem
        - quantidade_pesquisas_distintas
    ),
    ("ids_pesquisa_com_conflito", quantidade_conflitos_pesquisa),
    (
        "chamados_com_multiplas_pesquisas",
        quantidade_chamados_multiplas_pesquisas
    )
]

display(
    spark.createDataFrame(
        resumo_pesquisas,
        ["verificacao", "quantidade"]
    )
)

if quantidade_conflitos_pesquisa > 0:
    raise ValueError(
        "Existem id_pesquisa com registros diferentes. "
        "Investigue antes de continuar."
    )

df_distribuicao_notas = (
    df_pesquisa_satisfacao_origem
    .select(
        sf.when(
            sf.col("nota_atendimento").isNull(),
            "[nulo]"
        )
        .when(
            sf.trim(sf.col("nota_atendimento")) == "",
            "[vazio]"
        )
        .otherwise(
            sf.col("nota_atendimento")
        )
        .alias("nota_recebida")
    )
    .groupBy("nota_recebida")
    .count()
    .orderBy("nota_recebida")
)

display(df_distribuicao_notas)

### 11. Padronização das notas de satisfação

Preservamos o valor recebido em nota_atendimento_original.

Consideramos ausências os valores nulos, textos vazios e o texto NULL,
independentemente de maiúsculas ou minúsculas.

Consideramos válidas as notas inteiras de 1 a 5, após remover espaços
nas extremidades do texto. Outros conteúdos são classificados como
fora do domínio.

Criamos nota_atendimento como uma coluna numérica inteira, preenchida
somente para notas válidas. Notas ausentes ou inválidas permanecem
nulas nessa coluna, preservando o valor original para auditoria.

As pesquisas são mantidas mesmo quando sua nota está ausente ou
fora do domínio. As sinalizações permitem selecionar os registros
adequados para cada análise.

In [0]:
def converter_pesquisas(dados):
    return (
        dados
        .withColumnRenamed(
            "nota_atendimento",
            "nota_atendimento_original"
        )
        .withColumn(
            "nota_texto",
            sf.trim(sf.col("nota_atendimento_original"))
        )
        .withColumn(
            "nota_ausente",
            sf.col("nota_atendimento_original").isNull()
            | sf.lower(sf.col("nota_texto")).isin("", "null")
        )
        .withColumn(
            "nota_valida",
            sf.coalesce(
                sf.col("nota_texto").rlike(r"^[1-5]$"),
                sf.lit(False)
            )
        )
        .withColumn(
            "nota_fora_dominio",
            ~sf.col("nota_ausente")
            & ~sf.col("nota_valida")
        )
        .withColumn(
            "nota_atendimento",
            sf.when(
                sf.col("nota_valida"),
                sf.expr("try_cast(nota_texto as int)")
            )
        )
        .drop("nota_texto")
    )


df_pesquisas_tratadas = converter_pesquisas(
    df_pesquisas_distintas
)

### 12. Conferência das notas tratadas

Cada pesquisa deve pertencer a exatamente uma das categorias:
nota válida, nota ausente ou nota fora do domínio.

A soma dessas categorias deve corresponder à quantidade de pesquisas
após a verificação de duplicatas exatas.

Conferimos também se a quantidade de notas numéricas corresponde
à quantidade de notas classificadas como válidas.

In [0]:
df_resumo_notas = df_pesquisas_tratadas.agg(
    sf.count("*").alias("registros"),
    sf.sum(
        sf.col("nota_valida").cast("long")
    ).alias("notas_validas"),
    sf.sum(
        sf.col("nota_ausente").cast("long")
    ).alias("notas_ausentes"),
    sf.sum(
        sf.col("nota_fora_dominio").cast("long")
    ).alias("notas_fora_dominio"),
    sf.count("nota_atendimento").alias("notas_numericas")
)

display(df_resumo_notas)

metricas_notas = df_resumo_notas.first()

if metricas_notas["registros"] != quantidade_pesquisas_distintas:
    raise ValueError(
        "O tratamento alterou a quantidade de pesquisas."
    )

if metricas_notas["registros"] != (
    metricas_notas["notas_validas"]
    + metricas_notas["notas_ausentes"]
    + metricas_notas["notas_fora_dominio"]
):
    raise ValueError(
        "A classificação das notas não foi reconciliada."
    )

if metricas_notas["notas_numericas"] != metricas_notas["notas_validas"]:
    raise ValueError(
        "A quantidade de notas numéricas diverge das notas válidas."
    )

### Amostra das pesquisas tratadas

Exibimos cinco pesquisas para comparar a nota recebida com a nota
convertida e suas classificações de validade.

In [0]:
display(
    df_pesquisas_tratadas.select(
        "id_pesquisa",
        "id_chamado",
        "nota_atendimento_original",
        "nota_atendimento",
        "nota_valida",
        "nota_ausente",
        "nota_fora_dominio"
    ).limit(5)
)

### 13. Correspondência entre Pesquisas de Satisfação e Chamados

Reutilizamos a lista de chamados distintos e a função de correspondência
já utilizada em Custos.

Preservamos todas as pesquisas e acrescentamos chamado_encontrado,
indicando a existência do chamado na referência.

Verificamos a quantidade de pesquisas antes e depois do cruzamento,
as pesquisas sem correspondência e a cobertura de chamados com pesquisa.

A cobertura considera a existência de uma pesquisa, independentemente
da validade de sua nota. A disponibilidade de notas válidas será
considerada separadamente nas análises de satisfação.

In [0]:
df_pesquisas_com_correspondencia = (
    adicionar_correspondencia_chamados(
        df_pesquisas_tratadas
    )
)

quantidade_pesquisas_antes = metricas_notas["registros"]

quantidade_pesquisas_depois = (
    df_pesquisas_com_correspondencia.count()
)

quantidade_pesquisas_sem_chamado = (
    df_pesquisas_com_correspondencia
    .filter(~sf.col("chamado_encontrado"))
    .count()
)

quantidade_chamados_com_pesquisa = (
    df_pesquisas_com_correspondencia
    .filter(sf.col("chamado_encontrado"))
    .select("id_chamado")
    .distinct()
    .count()
)

df_chamados_sem_pesquisa = df_ids_chamados.join(
    df_pesquisas_tratadas.select("id_chamado").distinct(),
    on="id_chamado",
    how="left_anti"
)

resumo_correspondencia_pesquisas = [
    ("pesquisas_antes_cruzamento", quantidade_pesquisas_antes),
    ("pesquisas_apos_cruzamento", quantidade_pesquisas_depois),
    ("pesquisas_sem_chamado", quantidade_pesquisas_sem_chamado),
    ("chamados_com_pesquisa", quantidade_chamados_com_pesquisa),
    ("chamados_sem_pesquisa", df_chamados_sem_pesquisa.count())
]

display(
    spark.createDataFrame(
        resumo_correspondencia_pesquisas,
        ["verificacao", "quantidade"]
    )
)

if quantidade_pesquisas_antes != quantidade_pesquisas_depois:
    raise ValueError(
        "O cruzamento alterou a quantidade de pesquisas."
    )

### 14. Reexecução das transformações de Pesquisas de Satisfação

Reaplicamos a deduplicação, o tratamento das notas e a verificação
de correspondência sobre a mesma origem.

Comparamos os resultados nos dois sentidos, considerando os valores
e a multiplicidade das linhas. As colunas são alinhadas na mesma
ordem antes da comparação.

O resultado esperado é ausência de diferenças entre as execuções.
A verificação da gravação será realizada na publicação das tabelas.

In [0]:
df_pesquisas_reexecucao = adicionar_correspondencia_chamados(
    converter_pesquisas(
        df_pesquisa_satisfacao_origem.dropDuplicates(
            colunas_pesquisas
        )
    )
)

colunas_comparacao_pesquisas = sorted(
    df_pesquisas_com_correspondencia.columns
)

df_pesquisas_primeira_execucao = (
    df_pesquisas_com_correspondencia
    .select(*colunas_comparacao_pesquisas)
)

df_pesquisas_segunda_execucao = (
    df_pesquisas_reexecucao
    .select(*colunas_comparacao_pesquisas)
)

if (
    df_pesquisas_primeira_execucao.schema
    != df_pesquisas_segunda_execucao.schema
):
    raise ValueError(
        "A reexecução alterou a estrutura das pesquisas."
    )

diferencas_pesquisas = [
    (
        "linhas_ausentes",
        df_pesquisas_primeira_execucao
        .exceptAll(df_pesquisas_segunda_execucao)
        .count()
    ),
    (
        "linhas_adicionais",
        df_pesquisas_segunda_execucao
        .exceptAll(df_pesquisas_primeira_execucao)
        .count()
    )
]

display(
    spark.createDataFrame(
        diferencas_pesquisas,
        ["verificacao", "quantidade"]
    )
)

if any(quantidade > 0 for _, quantidade in diferencas_pesquisas):
    raise ValueError(
        "Foram encontradas diferenças na reexecução das pesquisas."
    )

df_pesquisa_satisfacao_silver = (
    df_pesquisas_com_correspondencia.select(
        "id_pesquisa",
        "id_chamado",
        "nota_atendimento",
        "nota_atendimento_original",
        "nota_valida",
        "nota_ausente",
        "nota_fora_dominio",
        "chamado_encontrado"
    )
)

### 15. Verificação de duplicidades em Chamados_Hora

Verificamos repetições exatas considerando todos os cinco campos
de negócio da base.

Após remover eventuais repetições exatas, verificamos se um mesmo
id_chamado ainda apresenta registros diferentes.

Caso existam versões conflitantes para um chamado, interrompemos
o processamento para investigação, sem escolher horários arbitrariamente.

In [0]:
colunas_chamados_hora = [
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento"
]

df_horarios_distintos = (
    df_chamados_hora_origem
    .dropDuplicates(colunas_chamados_hora)
)

quantidade_horarios_origem = df_chamados_hora_origem.count()

quantidade_horarios_distintos = df_horarios_distintos.count()

quantidade_conflitos_horarios = (
    df_horarios_distintos
    .groupBy("id_chamado")
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

resumo_duplicidades_horarios = [
    ("linhas_origem", quantidade_horarios_origem),
    (
        "repeticoes_exatas",
        quantidade_horarios_origem - quantidade_horarios_distintos
    ),
    ("ids_chamado_com_conflito", quantidade_conflitos_horarios)
]

display(
    spark.createDataFrame(
        resumo_duplicidades_horarios,
        ["verificacao", "quantidade"]
    )
)

if quantidade_conflitos_horarios > 0:
    raise ValueError(
        "Existem horários diferentes para o mesmo id_chamado. "
        "Investigue antes de continuar."
    )

### 16. Conversão dos horários

Preservamos os textos recebidos nas colunas com sufixo _original.

Reconhecemos o formato DD/MM/AAAA às HH:MM:SS e reorganizamos seus
componentes para o formato ISO antes da conversão.

Utilizamos timestamp_ntz para preservar os horários locais registrados,
sem atribuir ou converter um fuso horário não informado na fonte.

Utilizamos try_cast para que datas impossíveis ou conteúdos incompatíveis
sejam convertidos para nulo, mantendo o texto original para investigação.

Distinguimos ausência de informação e falha de conversão.
A ordem dos horários e os limites do período do case serão verificados
na etapa seguinte.

In [0]:
campos_horario = [
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento"
]


def converter_horarios(dados):
    resultado = dados

    padrao = (
        r"^(\d{2})/(\d{2})/(\d{4})"
        r"\s+às\s+(\d{2}:\d{2}:\d{2})$"
    )

    for campo in campos_horario:
        campo_original = f"{campo}_original"

        resultado = resultado.withColumnRenamed(
            campo,
            campo_original
        )

        texto = sf.trim(sf.col(campo_original))

        resultado = (
            resultado
            .withColumn(
                "horario_texto",
                sf.when(
                    texto.rlike(padrao),
                    sf.regexp_replace(
                        texto,
                        padrao,
                        "$3-$2-$1 $4"
                    )
                )
            )
            .withColumn(
                campo,
                sf.expr(
                    "try_cast(horario_texto as timestamp_ntz)"
                )
            )
            .drop("horario_texto")
        )

    return resultado


df_horarios_convertidos = converter_horarios(
    df_horarios_distintos
)

verificacoes_horarios = []

for campo in campos_horario:
    original = sf.col(f"{campo}_original")

    ausente = (
        original.isNull()
        | sf.lower(sf.trim(original)).isin("", "null")
    )

    invalido = (
        ~ausente
        & sf.col(campo).isNull()
    )

    verificacoes_horarios.extend(
        [
            sf.sum(
                ausente.cast("long")
            ).alias(f"{campo}_ausente"),
            sf.sum(
                invalido.cast("long")
            ).alias(f"{campo}_invalido")
        ]
    )

metricas_conversao_horarios = (
    df_horarios_convertidos
    .agg(*verificacoes_horarios)
    .first()
    .asDict()
)

display(
    spark.createDataFrame(
        list(metricas_conversao_horarios.items()),
        ["verificacao", "quantidade"]
    )
)

### 17. Cálculo dos tempos e coerência temporal

Calculamos, em segundos:
- Espera: início do atendimento menos abertura.
- Atendimento: finalização menos início do atendimento.
- Total: finalização menos abertura.

A subtração é realizada diretamente entre valores timestamp_ntz,
preservando os horários registrados sem conversão de fuso.
Como a fonte possui precisão de segundos, o intervalo é convertido
para um número inteiro de segundos.

Os campos com sufixo _calculado_segundos preservam os resultados,
inclusive negativos, para auditoria.

Sinalizamos início anterior à abertura, finalização anterior ao início
e finalização anterior à abertura.

Como critério conservador, os campos tempo_espera_segundos,
tempo_atendimento_segundos e tempo_total_segundos recebem valores
somente quando o respectivo intervalo existe e não há contradição
temporal conhecida no chamado.

Tempos negativos não são transformados em positivos.
Espera igual a zero é preservada.

O tempo total inclui espera e atendimento. Quando o início está
ausente, esse total pode estar disponível mesmo que os outros dois
intervalos não possam ser calculados.

In [0]:
def calcular_tempos_horarios(dados):
    resultado = dados

    intervalos = [
        (
            "espera",
            "hora_abertura_chamado",
            "hora_inicio_atendimento"
        ),
        (
            "atendimento",
            "hora_inicio_atendimento",
            "hora_finalizacao_atendimento"
        ),
        (
            "total",
            "hora_abertura_chamado",
            "hora_finalizacao_atendimento"
        )
    ]

    for nome, inicio, fim in intervalos:
        resultado = resultado.withColumn(
            f"tempo_{nome}_calculado_segundos",
            (sf.col(fim) - sf.col(inicio)).cast("long")
        )

    resultado = (
        resultado
        .withColumn(
            "inicio_antes_abertura",
            sf.coalesce(
                sf.col("tempo_espera_calculado_segundos") < 0,
                sf.lit(False)
            )
        )
        .withColumn(
            "finalizacao_antes_inicio",
            sf.coalesce(
                sf.col("tempo_atendimento_calculado_segundos") < 0,
                sf.lit(False)
            )
        )
        .withColumn(
            "finalizacao_antes_abertura",
            sf.coalesce(
                sf.col("tempo_total_calculado_segundos") < 0,
                sf.lit(False)
            )
        )
        .withColumn(
            "ordem_temporal_invalida",
            sf.col("inicio_antes_abertura")
            | sf.col("finalizacao_antes_inicio")
            | sf.col("finalizacao_antes_abertura")
        )
    )

    for nome, _, _ in intervalos:
        valor = sf.col(f"tempo_{nome}_calculado_segundos")

        resultado = resultado.withColumn(
            f"tempo_{nome}_segundos",
            sf.when(
                ~sf.col("ordem_temporal_invalida")
                & (valor >= 0),
                valor
            )
        )

    return resultado


df_horarios_com_tempos = calcular_tempos_horarios(
    df_horarios_convertidos
)

### 18. Conferência da coerência dos tempos

Contamos as inconsistências e a disponibilidade de cada intervalo.

Um chamado pode apresentar mais de uma inconsistência. Portanto,
as contagens individuais podem se sobrepor. A coluna
ordem_temporal_invalida identifica os chamados afetados sem
contá-los repetidamente.

Quando espera e atendimento estão disponíveis, verificamos se
sua soma corresponde ao tempo total.

Nesta conferência consideramos toda a base. O período do case
será sinalizado separadamente.

In [0]:
intervalos_parciais_disponiveis = (
    sf.col("tempo_espera_segundos").isNotNull()
    & sf.col("tempo_atendimento_segundos").isNotNull()
)

condicoes_tempos = {
    "inicio_antes_abertura":
        sf.col("inicio_antes_abertura"),
    "finalizacao_antes_inicio":
        sf.col("finalizacao_antes_inicio"),
    "finalizacao_antes_abertura":
        sf.col("finalizacao_antes_abertura"),
    "ordem_temporal_invalida":
        sf.col("ordem_temporal_invalida"),
    "espera_disponivel":
        sf.col("tempo_espera_segundos").isNotNull(),
    "atendimento_disponivel":
        sf.col("tempo_atendimento_segundos").isNotNull(),
    "tempo_total_disponivel":
        sf.col("tempo_total_segundos").isNotNull(),
    "espera_igual_a_zero":
        sf.col("tempo_espera_segundos") == 0,
    "tempos_com_soma_divergente":
        intervalos_parciais_disponiveis
        & (
            sf.col("tempo_total_segundos").isNull()
            | (
                sf.col("tempo_total_segundos")
                != (
                    sf.col("tempo_espera_segundos")
                    + sf.col("tempo_atendimento_segundos")
                )
            )
        )
}

agregacoes_tempos = [
    sf.count("*").alias("registros")
] + [
    sf.count(sf.when(condicao, 1)).alias(nome)
    for nome, condicao in condicoes_tempos.items()
]

metricas_tempos = (
    df_horarios_com_tempos
    .agg(*agregacoes_tempos)
    .first()
    .asDict()
)

display(
    spark.createDataFrame(
        list(metricas_tempos.items()),
        ["verificacao", "quantidade"]
    )
)

if metricas_tempos["registros"] != quantidade_horarios_distintos:
    raise ValueError(
        "O cálculo dos tempos alterou a quantidade de registros."
    )

if metricas_tempos["tempos_com_soma_divergente"] != 0:
    raise ValueError(
        "Foram encontradas divergências na soma dos tempos."
    )

### Amostras dos horários tratados

Exibimos cinco registros para visualizar a conversão dos horários
e os tempos tratados.

Em seguida, exibimos até cinco registros com ordem temporal inválida.
Esses exemplos mostram como as ocorrências foram preservadas e
sinalizadas, mantendo os tempos tratados nulos conforme a regra adotada.

In [0]:
df_visualizacao_horarios = df_horarios_com_tempos.select(
    "id_chamado",
    "hora_abertura_chamado_original",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "tempo_espera_calculado_segundos",
    "tempo_espera_segundos",
    "tempo_atendimento_calculado_segundos",
    "tempo_atendimento_segundos",
    "ordem_temporal_invalida"
)

print("Amostra geral:")
display(
    df_visualizacao_horarios.limit(5)
)

print("Exemplos com inconsistência temporal:")
display(
    df_visualizacao_horarios
    .filter(sf.col("ordem_temporal_invalida"))
    .limit(5)
)

### 19. Verificação do período do case

Adotamos a abertura como referência para enquadrar o chamado
no período informado: janeiro de 2025 a julho de 2026.

O intervalo inclui 01/01/2025 às 00:00:00 e termina antes
de 01/08/2026 às 00:00:00.

Preservamos os registros fora desse intervalo e acrescentamos
sinalizações para permitir o filtro nas análises.

Aberturas indisponíveis são identificadas separadamente.
O enquadramento pelo horário de abertura permite que um chamado
iniciado no último dia do período seja finalizado posteriormente.

In [0]:
data_inicio_case = "2025-01-01 00:00:00"
data_fim_case_exclusiva = "2026-08-01 00:00:00"


def sinalizar_periodo_horarios(dados):
    abertura = sf.col("hora_abertura_chamado")

    inicio = sf.lit(data_inicio_case).cast("timestamp_ntz")
    fim = sf.lit(data_fim_case_exclusiva).cast("timestamp_ntz")

    return (
        dados
        .withColumn(
            "abertura_fora_periodo",
            sf.coalesce(
                (abertura < inicio) | (abertura >= fim),
                sf.lit(False)
            )
        )
        .withColumn(
            "abertura_no_periodo",
            abertura.isNotNull()
            & ~sf.col("abertura_fora_periodo")
        )
    )


df_horarios_com_periodo = sinalizar_periodo_horarios(
    df_horarios_com_tempos
)

metricas_periodo = (
    df_horarios_com_periodo
    .agg(
        sf.count("*").alias("registros"),
        sf.count(
            sf.when(sf.col("abertura_no_periodo"), 1)
        ).alias("aberturas_no_periodo"),
        sf.count(
            sf.when(sf.col("abertura_fora_periodo"), 1)
        ).alias("aberturas_fora_periodo"),
        sf.count(
            sf.when(
                sf.col("hora_abertura_chamado").isNull(),
                1
            )
        ).alias("aberturas_indisponiveis")
    )
    .first()
    .asDict()
)

display(
    spark.createDataFrame(
        list(metricas_periodo.items()),
        ["verificacao", "quantidade"]
    )
)

if metricas_periodo["registros"] != (
    metricas_periodo["aberturas_no_periodo"]
    + metricas_periodo["aberturas_fora_periodo"]
    + metricas_periodo["aberturas_indisponiveis"]
):
    raise ValueError(
        "A classificação do período não foi reconciliada."
    )

### 20. Preparação da referência para conferir os horários

A base de chamados será utilizada apenas como referência, sem alterar sua tabela.

Selecionamos o identificador do chamado, o cliente e os tempos informados na origem. Removemos repetições idênticas dessa seleção para evitar a multiplicação de linhas no cruzamento.

Se um mesmo chamado apresentar informações diferentes na referência, interrompemos o processamento. Não escolhemos uma versão arbitrariamente.

Os identificadores permanecem como texto, preservando eventuais zeros à esquerda.

In [0]:
df_referencia_horarios = (
    df_chamados_referencia
    .select(
        "id_chamado",
        sf.col("id_cliente").alias("id_cliente_referencia"),
        sf.col("tempo_espera").alias(
            "tempo_espera_informado_original"
        ),
        sf.col("tempo_atendimento").alias(
            "tempo_atendimento_informado_original"
        )
    )
    .dropDuplicates()
)

quantidade_conflitos_referencia = (
    df_referencia_horarios
    .groupBy("id_chamado")
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

if quantidade_conflitos_referencia > 0:
    raise ValueError(
        "Existem chamados com informações conflitantes na referência."
    )

print(
    "Chamados com conflito na referência:",
    quantidade_conflitos_referencia
)

### 21. Correspondência entre horários e chamados

O cruzamento será realizado pelo id_chamado, mantendo todos os registros de Chamados_Hora.

Preservamos o id_cliente de cada origem em colunas distintas. Quando ambos estão preenchidos, verificamos se correspondem.

Uma divergência será sinalizada, sem substituir automaticamente nenhum identificador. A correção depende de uma definição sobre a fonte oficial do vínculo entre chamado e cliente.

Identificadores nulos, vazios ou contendo o texto "null" são considerados ausentes. Esses casos não participam da comparação entre clientes.

In [0]:
def identificador_ausente(nome_coluna):
    coluna = sf.col(nome_coluna)
    texto = sf.lower(sf.trim(coluna.cast("string")))

    return coluna.isNull() | texto.isin("", "null")


def cruzar_referencia_horarios(dados):
    referencia = df_referencia_horarios.withColumn(
        "chamado_encontrado",
        sf.lit(True)
    )

    return (
        dados
        .join(
            referencia,
            on="id_chamado",
            how="left"
        )
        .fillna({"chamado_encontrado": False})
        .withColumn(
            "cliente_comparavel",
            sf.col("chamado_encontrado")
            & ~identificador_ausente("id_cliente")
            & ~identificador_ausente("id_cliente_referencia")
        )
        .withColumn(
            "cliente_divergente",
            sf.col("cliente_comparavel")
            & ~sf.col("id_cliente").eqNullSafe(
                sf.col("id_cliente_referencia")
            )
        )
    )


df_horarios_com_correspondencia = cruzar_referencia_horarios(
    df_horarios_com_periodo
)

### 22. Validação dos vínculos e da quantidade de registros

O cruzamento deve preservar a quantidade de registros de Chamados_Hora.

Verificamos a cobertura nos dois sentidos: horários sem chamado correspondente e chamados sem registro de horário.

Para os clientes, cada registro deve pertencer a exatamente um grupo: correspondente, divergente ou não comparável.

Divergências de cliente permanecem sinalizadas para análise. Elas não justificam excluir o chamado nem alterar seu cliente automaticamente.

In [0]:
metricas_vinculos_horarios = (
    df_horarios_com_correspondencia
    .agg(
        sf.count("*").alias("horarios_apos_cruzamento"),

        sf.count(
            sf.when(~sf.col("chamado_encontrado"), 1)
        ).alias("horarios_sem_chamado"),

        sf.count(
            sf.when(~sf.col("cliente_comparavel"), 1)
        ).alias("clientes_nao_comparaveis"),

        sf.count(
            sf.when(sf.col("cliente_divergente"), 1)
        ).alias("clientes_divergentes"),

        sf.count(
            sf.when(
                sf.col("cliente_comparavel")
                & ~sf.col("cliente_divergente"),
                1
            )
        ).alias("clientes_correspondentes"),

        sf.count(
            sf.when(
                identificador_ausente("id_chamado"),
                1
            )
        ).alias("id_chamado_ausente_horarios")
    )
    .first()
    .asDict()
)

metricas_vinculos_horarios["horarios_antes_cruzamento"] = (
    df_horarios_com_periodo.count()
)

metricas_vinculos_horarios["chamados_distintos_referencia"] = (
    df_referencia_horarios.count()
)

metricas_vinculos_horarios["chamados_sem_horario"] = (
    df_referencia_horarios
    .select("id_chamado")
    .join(
        df_horarios_com_periodo.select("id_chamado"),
        on="id_chamado",
        how="left_anti"
    )
    .count()
)

metricas_vinculos_horarios["id_chamado_ausente_referencia"] = (
    df_referencia_horarios
    .filter(identificador_ausente("id_chamado"))
    .count()
)

display(
    spark.createDataFrame(
        list(metricas_vinculos_horarios.items()),
        ["verificacao", "quantidade"]
    )
)

if (
    metricas_vinculos_horarios["horarios_antes_cruzamento"]
    != metricas_vinculos_horarios["horarios_apos_cruzamento"]
):
    raise ValueError("O cruzamento alterou a quantidade de horários.")

total_classificacoes_cliente = sum(
    metricas_vinculos_horarios[nome]
    for nome in [
        "clientes_nao_comparaveis",
        "clientes_divergentes",
        "clientes_correspondentes"
    ]
)

if (
    total_classificacoes_cliente
    != metricas_vinculos_horarios["horarios_apos_cruzamento"]
):
    raise ValueError("A classificação dos clientes não fecha.")

if (
    metricas_vinculos_horarios["id_chamado_ausente_horarios"] > 0
    or metricas_vinculos_horarios["id_chamado_ausente_referencia"] > 0
):
    raise ValueError("Existem identificadores de chamado ausentes.")

### 23. Comparação dos tempos entre as fontes

Conforme o dicionário do case, os tempos informados em Chamados representam segundos. Nesta base, os valores preenchidos foram identificados como números inteiros.

Preservamos os valores originais e convertemos apenas textos que representam segundos inteiros. Valores ausentes e formatos inválidos recebem sinalizações distintas. Zero é um tempo válido.

Para os tempos derivados deste notebook, adotamos os eventos registrados em Chamados_Hora como fonte de cálculo. Os valores informados em Chamados permanecem disponíveis para comparação, sem substituir automaticamente horários ausentes.

A comparação de qualidade utiliza os campos calculados de auditoria, incluindo registros com ordem temporal inválida ou abertura fora do período. Isso permite identificar divergências sem ocultar essas ocorrências.

Um tempo é comparável quando existem tanto o valor informado convertido quanto o intervalo calculado. Quando não é comparável, a sinalização de divergência permanece nula, pois não é possível concluir se os valores coincidem.

A diferença em segundos corresponde a:
tempo recalculado − tempo informado.
Um resultado positivo indica que o tempo recalculado é maior.

Não utilizamos os tempos informados para preencher automaticamente horários ausentes nem para classificar abandono.

In [0]:
def comparar_tempos_informados(dados):
    resultado = dados

    for tipo in ["espera", "atendimento"]:
        prefixo = f"tempo_{tipo}"

        campo_original = f"{prefixo}_informado_original"
        campo_informado = f"{prefixo}_informado_segundos"
        campo_calculado = f"{prefixo}_calculado_segundos"

        texto = sf.trim(sf.col(campo_original))

        resultado = (
            resultado
            .withColumn(
                f"{prefixo}_informado_ausente",
                texto.isNull()
                | sf.lower(texto).isin("", "null")
            )
            .withColumn(
                campo_informado,
                sf.when(
                    ~sf.col(f"{prefixo}_informado_ausente")
                    & texto.rlike(r"^[+-]?[0-9]+$"),
                    sf.expr(
                        f"try_cast(trim({campo_original}) as bigint)"
                    )
                )
            )
            .withColumn(
                f"{prefixo}_informado_invalido",
                ~sf.col(f"{prefixo}_informado_ausente")
                & sf.col(campo_informado).isNull()
            )
            .withColumn(
                f"{prefixo}_informado_negativo",
                sf.coalesce(
                    sf.col(campo_informado) < 0,
                    sf.lit(False)
                )
            )
            .withColumn(
                f"{prefixo}_comparavel",
                sf.col(campo_informado).isNotNull()
                & sf.col(campo_calculado).isNotNull()
            )
            .withColumn(
                f"{prefixo}_divergente",
                sf.when(
                    sf.col(f"{prefixo}_comparavel"),
                    sf.col(campo_calculado)
                    != sf.col(campo_informado)
                )
            )
            .withColumn(
                f"diferenca_{prefixo}_segundos",
                sf.when(
                    sf.col(f"{prefixo}_comparavel"),
                    sf.col(campo_calculado)
                    - sf.col(campo_informado)
                )
            )
        )

    return resultado


df_horarios_com_comparacao = comparar_tempos_informados(
    df_horarios_com_correspondencia
)

### 24. Validação da comparação dos tempos

Verificamos separadamente os tempos de espera e de atendimento.

Cada valor informado deve ser classificado como ausente, inválido ou numérico. Valores negativos são sinalizados dentro do grupo numérico.

Cada registro deve ser classificado como comparável ou não comparável. Entre os comparáveis, os valores devem ser coincidentes ou divergentes.

As contagens devem fechar com o total de registros após o cruzamento. As divergências encontradas permanecem sinalizadas e não provocam exclusão de registros.

In [0]:
resumos = []

for tipo in ["espera", "atendimento"]:
    prefixo = f"tempo_{tipo}"

    comparavel = sf.col(f"{prefixo}_comparavel")
    divergente = sf.col(f"{prefixo}_divergente")

    condicoes = {
        "informados_ausentes": sf.col(
            f"{prefixo}_informado_ausente"
        ),
        "informados_invalidos": sf.col(
            f"{prefixo}_informado_invalido"
        ),
        "informados_numericos": sf.col(
            f"{prefixo}_informado_segundos"
        ).isNotNull(),
        "informados_negativos": sf.col(
            f"{prefixo}_informado_negativo"
        ),
        "comparaveis": comparavel,
        "coincidentes": comparavel & ~divergente,
        "divergentes": comparavel & divergente,
        "nao_comparaveis": ~comparavel
    }

    resumo = df_horarios_com_comparacao.agg(
        sf.lit(tipo).alias("tempo"),
        sf.count("*").alias("registros"),
        *[
            sf.count(sf.when(condicao, 1)).alias(nome)
            for nome, condicao in condicoes.items()
        ]
    )

    resumos.append(resumo)

df_resumo_comparacao_tempos = (
    resumos[0].unionByName(resumos[1])
)

display(df_resumo_comparacao_tempos)

for resumo in df_resumo_comparacao_tempos.collect():
    tipo = resumo["tempo"]

    if (
        resumo["registros"]
        != metricas_vinculos_horarios["horarios_apos_cruzamento"]
    ):
        raise ValueError(
            f"A comparação de {tipo} alterou a quantidade de registros."
        )

    total_classificacoes = (
        resumo["informados_ausentes"]
        + resumo["informados_invalidos"]
        + resumo["informados_numericos"]
    )

    if resumo["registros"] != total_classificacoes:
        raise ValueError(
            f"A classificação dos valores de {tipo} não fecha."
        )

    if (
        resumo["comparaveis"]
        != resumo["coincidentes"] + resumo["divergentes"]
    ):
        raise ValueError(
            f"A classificação dos comparáveis de {tipo} não fecha."
        )

    if (
        resumo["registros"]
        != resumo["comparaveis"] + resumo["nao_comparaveis"]
    ):
        raise ValueError(
            f"A cobertura da comparação de {tipo} não fecha."
        )

print("Verificações da comparação dos tempos concluídas.")

### 25. Estrutura final da Silver de Chamados_Hora

A tabela mantém um registro por id_chamado.

Selecionamos explicitamente as colunas finais para estabilizar sua estrutura e preservar a rastreabilidade dos tratamentos.

Mantemos os horários originais, os horários convertidos, os intervalos calculados para auditoria, os tempos tratados e as sinalizações de qualidade.

Os identificadores de cliente das duas fontes permanecem separados. As divergências não são corrigidas automaticamente.

Os campos terminados em "_calculado_segundos" permitem auditar os cálculos, inclusive resultados negativos. Os campos "tempo_espera_segundos", "tempo_atendimento_segundos" e "tempo_total_segundos" seguem as regras de validade temporal definidas anteriormente.

Nenhum registro é excluído por inconsistência temporal, divergência de cliente ou abertura fora do período.

Validamos que a seleção preserva a quantidade de registros e que id_chamado está preenchido e é único.

In [0]:
colunas_chamados_hora_silver = [
    "id_chamado",
    "id_cliente",
    "id_cliente_referencia",
    "chamado_encontrado",
    "cliente_comparavel",
    "cliente_divergente",

    "hora_abertura_chamado",
    "hora_abertura_chamado_original",
    "hora_inicio_atendimento",
    "hora_inicio_atendimento_original",
    "hora_finalizacao_atendimento",
    "hora_finalizacao_atendimento_original",

    "abertura_no_periodo",
    "abertura_fora_periodo",
    "inicio_antes_abertura",
    "finalizacao_antes_inicio",
    "finalizacao_antes_abertura",
    "ordem_temporal_invalida",

    "tempo_espera_segundos",
    "tempo_espera_calculado_segundos",
    "tempo_atendimento_segundos",
    "tempo_atendimento_calculado_segundos",
    "tempo_total_segundos",
    "tempo_total_calculado_segundos",

    "tempo_espera_informado_original",
    "tempo_espera_informado_segundos",
    "tempo_espera_informado_ausente",
    "tempo_espera_informado_invalido",
    "tempo_espera_informado_negativo",
    "tempo_espera_comparavel",
    "tempo_espera_divergente",
    "diferenca_tempo_espera_segundos",

    "tempo_atendimento_informado_original",
    "tempo_atendimento_informado_segundos",
    "tempo_atendimento_informado_ausente",
    "tempo_atendimento_informado_invalido",
    "tempo_atendimento_informado_negativo",
    "tempo_atendimento_comparavel",
    "tempo_atendimento_divergente",
    "diferenca_tempo_atendimento_segundos"
]

df_chamados_hora_silver = (
    df_horarios_com_comparacao
    .select(*colunas_chamados_hora_silver)
)

metricas_chamados_hora_silver = (
    df_chamados_hora_silver
    .agg(
        sf.count("*").alias("registros"),
        sf.countDistinct("id_chamado").alias(
            "ids_chamado_distintos"
        ),
        sf.count(
            sf.when(
                identificador_ausente("id_chamado"),
                1
            )
        ).alias("ids_chamado_ausentes")
    )
    .first()
    .asDict()
)

metricas_chamados_hora_silver["quantidade_colunas"] = len(
    df_chamados_hora_silver.columns
)

display(
    spark.createDataFrame(
        list(metricas_chamados_hora_silver.items()),
        ["verificacao", "quantidade"]
    )
)

if (
    metricas_chamados_hora_silver["registros"]
    != quantidade_horarios_distintos
):
    raise ValueError(
        "A estrutura final alterou a quantidade de registros."
    )

if metricas_chamados_hora_silver["ids_chamado_ausentes"] > 0:
    raise ValueError(
        "Existem identificadores de chamado ausentes."
    )

if (
    metricas_chamados_hora_silver["registros"]
    != metricas_chamados_hora_silver["ids_chamado_distintos"]
):
    raise ValueError(
        "Existem identificadores de chamado repetidos na Silver."
    )

### 26. Verificação de reexecução das transformações

Reconstruímos o resultado desde os dados de origem, mantendo a mesma referência de chamados e as mesmas regras de tratamento.

O novo resultado deve apresentar o mesmo esquema e exatamente os mesmos registros, incluindo as sinalizações de qualidade e os valores nulos.

Comparamos os resultados nos dois sentidos para identificar linhas ausentes, adicionais ou com valores diferentes. A comparação considera também a quantidade de ocorrências de cada linha.

Este teste verifica a estabilidade das transformações em memória. A reexecução da gravação nas tabelas Delta será verificada na etapa de publicação.

In [0]:
def construir_silver_chamados_hora(dados_origem):
    resultado = dados_origem.dropDuplicates(
        colunas_chamados_hora
    )

    resultado = converter_horarios(resultado)
    resultado = calcular_tempos_horarios(resultado)
    resultado = sinalizar_periodo_horarios(resultado)
    resultado = cruzar_referencia_horarios(resultado)
    resultado = comparar_tempos_informados(resultado)

    return resultado.select(
        *colunas_chamados_hora_silver
    )


df_horarios_reexecucao = construir_silver_chamados_hora(
    df_chamados_hora_origem
)

if (
    df_chamados_hora_silver.schema
    != df_horarios_reexecucao.schema
):
    raise ValueError(
        "O esquema foi alterado na reexecução."
    )

linhas_ausentes_horarios = (
    df_chamados_hora_silver
    .exceptAll(df_horarios_reexecucao)
    .count()
)

linhas_adicionais_horarios = (
    df_horarios_reexecucao
    .exceptAll(df_chamados_hora_silver)
    .count()
)

display(
    spark.createDataFrame(
        [
            ("linhas_ausentes", linhas_ausentes_horarios),
            ("linhas_adicionais", linhas_adicionais_horarios)
        ],
        ["verificacao", "quantidade"]
    )
)

if (
    linhas_ausentes_horarios > 0
    or linhas_adicionais_horarios > 0
):
    raise ValueError(
        "A reexecução produziu registros diferentes."
    )

print(
    "Reexecução das transformações de Chamados_Hora "
    "verificada sem diferenças."
)

### 27. Conferência antes da publicação

Definimos explicitamente as três tabelas de destino desta contribuição.

Validamos o preenchimento e a unicidade da chave de cada base:
id_custo, id_pesquisa e id_chamado, respectivamente.

Também verificamos o preenchimento de id_chamado nas três bases.
Um chamado pode possuir diferentes pesquisas ou registros de custo;
por isso, a unicidade é verificada pela chave própria de cada base.

Como os arquivos do case possuem registros, interrompemos a
publicação caso algum resultado esteja vazio.

In [0]:
publicacoes_silver = [
    (
        "custos",
        f"{esquema_silver}.tb_custos",
        df_custos_silver,
        "id_custo"
    ),
    (
        "pesquisa_satisfacao",
        f"{esquema_silver}.tb_pesquisa_satisfacao",
        df_pesquisa_satisfacao_silver,
        "id_pesquisa"
    ),
    (
        "chamados_hora",
        f"{esquema_silver}.tb_chamados_hora",
        df_chamados_hora_silver,
        "id_chamado"
    )
]

resumo_pre_publicacao = []

for nome_base, nome_tabela, dados, chave in publicacoes_silver:
    metricas = dados.agg(
        sf.count("*").alias("registros"),
        sf.countDistinct(chave).alias("ids_distintos"),
        sf.count(
            sf.when(
                identificador_ausente(chave)
                | identificador_ausente("id_chamado"),
                1
            )
        ).alias("registros_com_id_ausente")
    ).first()

    if (
        metricas["registros"] == 0
        or metricas["registros"] != metricas["ids_distintos"]
        or metricas["registros_com_id_ausente"] > 0
    ):
        raise ValueError(
            f"Falha na validação das chaves de {nome_base}."
        )

    resumo_pre_publicacao.append(
        (nome_base, nome_tabela, metricas["registros"])
    )

display(
    spark.createDataFrame(
        resumo_pre_publicacao,
        ["base", "tabela_destino", "registros"]
    )
)

### 28. Publicação e conferência das tabelas Silver

Gravamos os resultados em tabelas Delta utilizando recarga integral,
adequada ao conjunto fixo de arquivos deste case.

Após cada gravação, registramos a versão Delta produzida e lemos
essa versão para conferir os dados persistidos.

Verificamos nomes e tipos das colunas e comparamos os registros
nos dois sentidos, considerando valores nulos e repetições.

A comparação alinha a ordem das colunas antes de utilizar exceptAll.

Em Custos, verificamos também se a soma gravada corresponde ao
total reconciliado anteriormente, incluindo os negativos preservados.

In [0]:
def comparar_conteudo(esperado, obtido, contexto):
    if set(esperado.columns) != set(obtido.columns):
        raise ValueError(
            f"Colunas diferentes em {contexto}."
        )

    obtido = obtido.select(*esperado.columns)

    if esperado.dtypes != obtido.dtypes:
        raise ValueError(
            f"Tipos de colunas diferentes em {contexto}."
        )

    linhas_ausentes = esperado.exceptAll(obtido).count()
    linhas_adicionais = obtido.exceptAll(esperado).count()

    if linhas_ausentes > 0 or linhas_adicionais > 0:
        raise ValueError(
            f"Diferenças em {contexto}: "
            f"{linhas_ausentes} linhas ausentes e "
            f"{linhas_adicionais} linhas adicionais."
        )

    return linhas_ausentes, linhas_adicionais


def gravar_silver(dados, nome_tabela):
    (
        dados.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

    return int(
        spark.sql(
            f"DESCRIBE HISTORY {nome_tabela} LIMIT 1"
        ).first()["version"]
    )


spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {esquema_silver}"
)

versoes_primeira_gravacao = {}
resumo_gravacao = []

for nome_base, nome_tabela, dados, chave in publicacoes_silver:
    versao = gravar_silver(dados, nome_tabela)

    dados_gravados = (
        spark.read
        .option("versionAsOf", versao)
        .table(nome_tabela)
    )

    ausentes, adicionais = comparar_conteudo(
        dados,
        dados_gravados,
        nome_tabela
    )

    if nome_base == "custos":
        total_custos_gravado = somar_custos(dados_gravados)

        if total_custos_gravado != total_custos_sem_duplicatas:
            raise ValueError(
                "A soma dos custos gravados divergiu."
            )

    versoes_primeira_gravacao[nome_tabela] = versao

    resumo_gravacao.append(
        (
            nome_base,
            dados_gravados.count(),
            ausentes,
            adicionais
        )
    )

display(
    spark.createDataFrame(
        resumo_gravacao,
        [
            "base",
            "registros_gravados",
            "linhas_ausentes",
            "linhas_adicionais"
        ]
    )
)

print(f"Soma dos custos gravados: {total_custos_gravado}")

### 29. Verificação de reexecução da gravação

Utilizamos os resultados reconstruídos nas verificações de reexecução
de Custos, Pesquisas de Satisfação e Chamados_Hora.

Antes de regravar, verificamos se esses resultados correspondem
à primeira versão persistida.

Repetimos a gravação e comparamos as versões Delta anteriores
e posteriores, verificando tipos, valores e multiplicidade das linhas.

Para o mesmo lote de entrada e as mesmas regras, o conteúdo deve
permanecer igual. O número da versão Delta pode aumentar, pois
a nova gravação é registrada no histórico.

In [0]:
reexecucoes_silver = {
    "custos": df_custos_reexecucao.select(
        *df_custos_silver.columns
    ),
    "pesquisa_satisfacao": df_pesquisas_reexecucao.select(
        *df_pesquisa_satisfacao_silver.columns
    ),
    "chamados_hora": df_horarios_reexecucao.select(
        *df_chamados_hora_silver.columns
    )
}

resumo_reexecucao_gravacao = []

for nome_base, nome_tabela, dados, chave in publicacoes_silver:
    versao_antes = versoes_primeira_gravacao[nome_tabela]

    dados_antes = (
        spark.read
        .option("versionAsOf", versao_antes)
        .table(nome_tabela)
    )

    dados_reexecucao = reexecucoes_silver[nome_base]

    comparar_conteudo(
        dados_antes,
        dados_reexecucao,
        f"reconstrução de {nome_base}"
    )

    versao_depois = gravar_silver(
        dados_reexecucao,
        nome_tabela
    )

    dados_depois = (
        spark.read
        .option("versionAsOf", versao_depois)
        .table(nome_tabela)
    )

    ausentes, adicionais = comparar_conteudo(
        dados_antes,
        dados_depois,
        f"regravação de {nome_base}"
    )

    resumo_reexecucao_gravacao.append(
        (
            nome_base,
            versao_antes,
            versao_depois,
            dados_antes.count(),
            dados_depois.count(),
            ausentes,
            adicionais
        )
    )

display(
    spark.createDataFrame(
        resumo_reexecucao_gravacao,
        [
            "base",
            "versao_antes",
            "versao_depois",
            "registros_antes",
            "registros_depois",
            "linhas_ausentes",
            "linhas_adicionais"
        ]
    )
)

print("Reexecução da gravação verificada sem diferenças.")

### Escopo e estágio do desenvolvimento

Esta contribuição contempla o tratamento e a publicação das bases
de Custos, Pesquisa de Satisfação e Chamados_Hora na camada Silver.

A leitura utiliza as tabelas Bronze oficiais do projeto.

Tabelas publicadas:
- vcredit_medallion.silver.tb_custos
- vcredit_medallion.silver.tb_pesquisa_satisfacao
- vcredit_medallion.silver.tb_chamados_hora

Foram verificadas duplicidades, conflitos e preenchimento de
identificadores, correspondência entre bases, reconciliação dos
custos, validade das notas e coerência dos horários.

A publicação utiliza recarga integral em Delta. Foram conferidos
os nomes e tipos das colunas, o conteúdo persistido e a soma dos custos.

A reexecução das transformações e da gravação foi validada.
As duas versões persistidas apresentaram os mesmos registros,
sem linhas ausentes ou adicionais.

Cuidados para consumo:
- Custos preserva os valores negativos e suas sinalizações.
- Pesquisas mantém uma linha por id_pesquisa. Um chamado pode
  possuir mais de uma pesquisa.
- A coluna nota_valida identifica as notas elegíveis para análises
  que utilizam a escala de 1 a 5.
- Os tempos tratados são os campos terminados em "_segundos",
  sem o termo "_calculado".
- A coluna abertura_no_periodo permite selecionar o período do case.
- Divergências de cliente permanecem sinalizadas para definição
  da fonte adequada nas análises por cliente.